# 011 — per-site case-study designs & analysis folders

For each of the 60 WP1 sites this notebook:

1. designs a **3-storey MDOF CBF** for the site (Soil Class A, local `S_alpha,475`),
   in parallel, using the existing `standes` design algorithm;
2. builds the **MDOF analysis folder** (structural model + modal + cyclic pushover +
   FEMA P695 IDA configs);
3. writes the **cyclic pushover and IDA batch launchers**.

Designs are saved **in-repo** under `casestudy_designs_site_specific`; the per-site design
base-shear coefficient `Vb_coeff` is recorded in `site_designs_summary.csv`. Analysis folders
are written under `DEST_ROOT` (set below), organised `DEST_ROOT/site_{ii}/mdof/`.

## Dependencies

**Upstream — only `001-site_selection`.** Sites are read from `sites.csv`; this notebook
needs *no* ground-motion records and *no* record selection, so it can run early enough to
feed `017`.

**Downstream.** `site_designs_summary.csv` (written by §2) is required by
`017-disagg_imls_for_msa_stripes`, which needs the per-site `Vb_coeff` to estimate stripe
collapse probabilities. **Run this notebook before `017`.**

**MSA wiring lives in `050_setup_msa_runs_for_complete_sites`, not here.** This notebook
deliberately stops at the structural models. Copying the stripe pickles, writing
`config_msa_{IM}.py` and generating the MSA batch launchers all require the record selection
(`033`/`036`) and the converted records (`042`), so they happen in `050`, which adds those
files into the `mdof/` folders built here.

## External analyses

Building the folders is cheap file generation. The cyclic pushover and IDA batch launchers
written by the last section must be run separately — they are not launched from this
notebook. The FEMA P695 IDA is by far the most expensive step.

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0. Setup & parameters

In [20]:
import os
import json
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from phd_project.config import config

from phd_project.scripts.case_study_design_scripts.design_site_mdof import (
    design_sites_parallel,
    default_n_workers,
)
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_primary_modes_from_design_file,
    get_n_damping_modes_from_design_file,
)
from phd_project.scripts.loading_protocols import FEMA_461_loading_protocol
from phd_project.scripts.cpo_du import resolve_cpo_du, audit_cpo_du, existing_cpo_du
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_nlcbf_model,
    copy_analysis_config,
    configure_batch_run_file,
    copy_batch_ida_buildings,
    copy_file,
)

cfg = config.load_config()

In [21]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Destination root for the (large) analysis folders -- set this to your external
# drive. Folders are written as DEST_ROOT/site_{ii}/mdof/.
DEST_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]

# --- structure / design (fixed; matches 01_design_case_study_structures.py) ---
SITE_CATEGORY = "A"        # soil class for the MDOF design
N_STOREYS = 3
DUCTILITY_CLASS = 2
STOREY_HEIGHT = 3500       # mm
BAY_WIDTH = 7000           # mm
MAX_DESIGN_ITERS = 15

# --- analysis ---
# NOTE: the MSA parameters (GM_SETS, MAX_N_RECORDS, STRIPE_ORDER_ASCENDING) live in
# 050_setup_msa_runs_for_complete_sites, which owns the MSA wiring.
DAMPING_RATIO = 0.05
MDOF_DRIFT_LIMIT = 0.2     # MDOF collapse drift limit

# --- parallelism ---
N_WORKERS = default_n_workers()    # max(cpu_count - 3, 1)

# Restrict to a subset of sites while testing (e.g. [0, 1, 2]); None = all sites.
LIMIT_SITES = None

S_ALPHA_COLUMN = "S_alpha,475"

# --- cyclic pushover (FEMA 461) ---
CPO_U_MAX = 250          # mm, peak cyclic amplitude (+/-)
CPO_N_STEPS = 12         # FEMA 461 amplitude steps (12 -> matches example sequence)
CPO_DU = 0.2             # mm, base displacement ramp step (dU_max)
CPO_DISPLACEMENTS = np.round(
    np.append(FEMA_461_loading_protocol(CPO_U_MAX, CPO_N_STEPS), 0), 3
).tolist()

# Per-folder dU overrides, {key: dU} where key = "site_{ii}/mdof" (the folder path
# relative to DEST_ROOT). The audit cell in section 1 prints divergent values ready to
# paste here. See phd_project/scripts/cpo_du.py.
CPO_DU_OVERRIDES = {}
# Keep a finer dU already present on disk instead of overwriting it with CPO_DU.
# Leave True unless you deliberately want to re-tune from scratch.
PRESERVE_TUNED_DU = True

# --- IDA (FEMA P695 far-field set) ---
GM_JSON_SRC = "D:/gm_records_p695"    # folder holding the record JSONs
FEMAP695_RECORDS = [
    "fema_p695_120111.json", "fema_p695_120121.json", "fema_p695_120411.json",
    "fema_p695_120521.json", "fema_p695_120611.json", "fema_p695_120621.json",
    "fema_p695_120711.json", "fema_p695_120721.json", "fema_p695_120811.json",
    "fema_p695_120821.json", "fema_p695_120911.json", "fema_p695_120921.json",
    "fema_p695_121011.json", "fema_p695_121021.json", "fema_p695_121111.json",
    "fema_p695_121211.json", "fema_p695_121221.json", "fema_p695_121321.json",
    "fema_p695_121411.json", "fema_p695_121421.json", "fema_p695_121511.json",
    "fema_p695_121711.json",
]

# --- batch launchers ---
# po / cpo / modal / ida launchers all land here; the folder path encodes the
# system (mdof) and storeys (3s), so the filenames are just the analysis type.
BATCH_ROOT = (Path(cfg["scripts"]["batch_run_analyses"])
                / "casestudy_sites" / "3s" / "mdof")

print(f"N_WORKERS = {N_WORKERS}; DEST_ROOT = {DEST_ROOT}")

N_WORKERS = 58; DEST_ROOT = D:\07_wp1_casestudy_sites


## 1. Discover sites

Sites come straight from `sites.csv` (written by `001-site_selection`): the positional
index `ii` *is* the row index (`.iloc[ii]`), which is the convention the rest of the
project uses (`site_{ii}` folders, `site_{ii}__stripe_{n}__gm_selection.pickle`).

This used to derive the site list from the record-selection pickle filenames, which made
the notebook depend on `033`/`036` and so impossible to run before `017`. Reading
`sites.csv` decouples it: all 60 sites are designed regardless of how many have records
selected yet, which is what `017` needs — it estimates stripe collapse probabilities for
every site.

In [22]:
sites_df = pd.read_csv(cfg["results"]["selected_sites_csv"])

all_sites = list(range(len(sites_df)))
if LIMIT_SITES is not None:
    all_sites = [s for s in all_sites if s in set(LIMIT_SITES)]

site_S_alpha = {ii: float(sites_df.iloc[ii][S_ALPHA_COLUMN]) for ii in all_sites}
site_tag = {ii: f"{N_STOREYS}s_cbf_dc{DUCTILITY_CLASS}_site{ii}" for ii in all_sites}

print(f"{len(all_sites)} sites (from {Path(cfg['results']['selected_sites_csv']).name}); "
      f"S_alpha range {min(site_S_alpha.values()):.3f} -> {max(site_S_alpha.values()):.3f}")

60 sites (from sites.csv); S_alpha range 0.134 -> 1.132


### Cyclic-pushover `dU` audit

Some models need a finer displacement increment than the `CPO_DU` default to converge, so the
value in use can vary per folder. Rebuilding a folder rewrites its `config_cyclic_pushover.py`
and would throw that tuning away — with `PRESERVE_TUNED_DU = True` (the default) any tuned `dU`
already on disk is kept instead. This audit reports the divergent `mdof`
folders and prints a snippet ready to paste into `CPO_DU_OVERRIDES` to pin them explicitly.

In [23]:
cpo_items = [
    (f"site_{ii}/mdof", DEST_ROOT / f"site_{ii}" / "mdof" / "config_cyclic_pushover.py")
    for ii in all_sites
]
divergent = audit_cpo_du(cpo_items, CPO_DU, CPO_DU_OVERRIDES)

if divergent:
    print(f"{len(divergent)} existing config(s) use a dU different from the default "
          f"({CPO_DU}):")
    for key, du in sorted(divergent.items()):
        print(f"   {key:24s} dU = {du}")
    print("\nPRESERVE_TUNED_DU =", PRESERVE_TUNED_DU,
          "-> these will be KEPT." if PRESERVE_TUNED_DU else "-> these will be OVERWRITTEN.")
    print("\nTo pin these explicitly, paste into CPO_DU_OVERRIDES:")
    print("CPO_DU_OVERRIDES = {")
    for key, du in sorted(divergent.items()):
        print(f'    "{key}": {du},')
    print("}")
else:
    print("no divergent dU values found")

no divergent dU values found


## 2. Design the MDOFs (parallel + tqdm)

Each MDOF is designed with the existing `standes` algorithm (unchanged) at Soil
Class A and the site's `S_alpha,475`. Designs are written **in-repo**.

In [24]:
design_root = Path(cfg["models"]["casestudy_designs_site_specific"])
design_root.mkdir(parents=True, exist_ok=True)
summary_path = design_root / "site_designs_summary.csv"

jobs = [(site_S_alpha[ii], site_tag[ii], design_root) for ii in all_sites]

# only (re)design structures whose tag is not already in the summary csv
existing_df = pd.read_csv(summary_path, index_col="tag") if summary_path.exists() else None
done_tags = set(existing_df.index) if existing_df is not None else set()
pending_jobs = [j for j in jobs if j[1] not in done_tags]

if pending_jobs:
    print(f"designing {len(pending_jobs)} of {len(jobs)} structures "
          f"({len(jobs) - len(pending_jobs)} already in {summary_path.name})")
    design_results = design_sites_parallel(
        pending_jobs,
        n_workers=N_WORKERS,
        site_category=SITE_CATEGORY,
        ductility_class=DUCTILITY_CLASS,
        storey_height=STOREY_HEIGHT,
        n_storeys=N_STOREYS,
        max_iters=MAX_DESIGN_ITERS,
    )
    new_df = pd.DataFrame(design_results).set_index("tag")
    if existing_df is not None:
        existing_df = existing_df.drop(index=new_df.index, errors="ignore")
        designs_df = pd.concat([existing_df, new_df])
    else:
        designs_df = new_df
    designs_df = designs_df.sort_index()
    designs_df.to_csv(summary_path)
else:
    print(f"all {len(jobs)} structures already in {summary_path.name}; skipping design")
    designs_df = existing_df.sort_index()

designs = designs_df.to_dict("index")

failed = designs_df[~designs_df["success"].fillna(False)]
if len(failed):
    print(f"WARNING: {len(failed)} designs did not succeed:\n{failed.index.tolist()}")

designs_df[["S_alpha_RP", "Vb_coeff", "Wt", "T", "q_design", "success"]]

all 60 structures already in site_designs_summary.csv; skipping design


,S_alpha_RP,Vb_coeff,Wt,T,q_design,success
tag,,,,,,
3s_cbf_dc2_site0,0.209594,0.028540,3061800.0,0.719978,2.040000,True
3s_cbf_dc2_site1,0.273973,0.039123,3061800.0,0.686561,2.040000,True
3s_cbf_dc2_site10,0.394579,0.084847,3061800.0,0.581311,2.040000,True
3s_cbf_dc2_site11,0.355457,0.076434,3061800.0,0.581311,2.040000,True
3s_cbf_dc2_site12,0.287359,0.041914,3061800.0,0.672157,2.040000,True
3s_cbf_dc2_site13,0.281817,0.041105,3061800.0,0.672157,2.040000,True
3s_cbf_dc2_site14,0.313111,0.046007,3061800.0,0.667227,2.040000,True
3s_cbf_dc2_site15,0.319417,0.039983,3061800.0,0.639112,2.500000,True
3s_cbf_dc2_site16,0.244726,0.034946,3061800.0,0.686561,2.040000,True


## 3. Design output dataset

Flatten the design file of each site's structure into a single tidy dataframe -- one row per
site, with the design base shear, seismic mass, behaviour factors, spectrum parameters and
storey forces/shears as columns, plus the `site_idx` and its `lat`/`lon` from `sites.csv`.

This only reads the design files written in section 2 (no OpenSees analysis needed), keeping
just the sites that designed successfully. The dataframe is pickled to
`cfg["proc_data"]["site_specific_dataset"]`.

In [25]:
# Flatten each design file into one row: mapping of {column: nested-json path}.
design_out_parameter_map = {
    "roof_height": "['structure']['level_coordinates'][-1]",
    "V_wind_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['uls_wind_base_shear']",
    "T1_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['period']",
    "T1_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['period']",
    "T1_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['period']",
    "gravity_frame_elastic_baseshear": "['seismic_design_outputs']['gravity_frame_elastic_baseshear']",
    "gravity_design_equivalent_seismic_baseshear": "['seismic_design_outputs']['gravity_design_equivalent_seismic_baseshear']",
    "seismic_mass": "['seismic_design_outputs']['seismic_mass']",
    "ductility_class": "['seismic_design_outputs']['ductility_class']",
    "vertical_regularity": "['seismic_design_outputs']['vertical_regularity']",
    "q_design": "['seismic_design_outputs']['q_design']",
    "q_D": "['seismic_design_outputs']['q_D']",
    "q_R": "['seismic_design_outputs']['q_R']",
    "q_S": "['seismic_design_outputs']['q_S']",
    "q_max": "['seismic_design_outputs']['q_max']",
    "design_period": "['seismic_design_outputs']['design_period']",
    "design_spectral_acceleration": "['seismic_design_outputs']['design_spectral_acceleration']",
    "design_baseshear": "['seismic_design_outputs']['design_baseshear']",
    "lambda": "['seismic_design_outputs']['lambda']",
    "S_alpha_RP": "['seismic_design_outputs']['spectrum_parameters']['S_alpha_RP']",
    "S_beta_RP": "['seismic_design_outputs']['spectrum_parameters']['S_beta_RP']",
    "S_alpha": "['seismic_design_outputs']['spectrum_parameters']['S_alpha']",
    "S_beta": "['seismic_design_outputs']['spectrum_parameters']['S_beta']",
    "T_beta": "['seismic_design_outputs']['spectrum_parameters']['T_beta']",
    "T_A": "['seismic_design_outputs']['spectrum_parameters']['T_A']",
    "T_B": "['seismic_design_outputs']['spectrum_parameters']['T_B']",
    "T_C": "['seismic_design_outputs']['spectrum_parameters']['T_C']",
    "T_D": "['seismic_design_outputs']['spectrum_parameters']['T_D']",
    "F_alpha": "['seismic_design_outputs']['spectrum_parameters']['F_alpha']",
    "F_beta": "['seismic_design_outputs']['spectrum_parameters']['F_beta']",
    "F_T": "['seismic_design_outputs']['spectrum_parameters']['F_T']",
    "F_A": "['seismic_design_outputs']['spectrum_parameters']['F_A']",
    "site_category": "['seismic_design_outputs']['spectrum_parameters']['site_category']",
    "S_delta": "['seismic_design_outputs']['spectrum_parameters']['S_delta']",
    "delta": "['seismic_design_outputs']['spectrum_parameters']['delta']",
    "seismic_action_class": "['seismic_design_outputs']['spectrum_parameters']['seismic_action_class']",
}


def read_out_value(data, path):
    """Pull a value out of the nested design dict, returning NaN if the path is absent.

    `path` is a string of chained subscripts, e.g. "['structure']['level_coordinates'][-1]".
    """
    try:
        return eval("data" + path)
    except Exception:
        return np.nan


MAX_STOREYS = 7  # width of the seismic_F* / seismic_V* columns

building_data_dicts = []
for tag in [t for t in designs if designs[t]["success"]]:
    with open(designs[tag]["design_out_json"], "r") as f:
        data = json.load(f)

    building_data = {"name": tag, "n_storeys": int(tag[0])}
    for df_tag, dd_path in design_out_parameter_map.items():
        building_data[df_tag] = read_out_value(data, dd_path)

    # design base-shear coefficient Vb/Wt (Wt = seismic_mass * g), carried straight
    # from the design summary so downstream notebooks read it without recomputing.
    # See phd_project/scripts/case_study_design_scripts/design_site_mdof.py.
    building_data["Vb"] = designs[tag].get("Vb")
    building_data["Wt"] = designs[tag].get("Wt")
    building_data["Vb_coeff"] = designs[tag].get("Vb_coeff")

    # design storey forces and the cumulative storey shears
    storey_forces = data["seismic_design_outputs"]["storey_forces"]
    storey_shears = np.cumsum(storey_forces)
    for ii in range(MAX_STOREYS):
        building_data[f"seismic_F{ii + 1}"] = (
            storey_forces[ii] if ii < len(storey_forces) else np.nan
        )
        building_data[f"seismic_V{ii + 1}"] = (
            storey_shears[ii] if ii < len(storey_shears) else np.nan
        )

    building_data_dicts.append(building_data)

building_data_df = pd.DataFrame(building_data_dicts)

# tie each row back to its WP1 site (tag ends in "site{ii}") and attach coordinates
building_data_df["site_idx"] = building_data_df["name"].str.split("site").str[-1].astype(int)
site_coords = sites_df[["lat", "lon"]].copy()
site_coords["site_idx"] = site_coords.index
building_data_df = building_data_df.merge(site_coords, on="site_idx", how="left")

building_data_df = building_data_df.sort_values(
    by=["n_storeys", "S_alpha_RP"]
).reset_index(drop=True)

dataset_path = cfg["proc_data"]["site_specific_dataset"]
with open(dataset_path, "wb") as f:
    pickle.dump(building_data_df, f)
print(f"wrote {dataset_path}  ({len(building_data_df)} rows)")

building_data_df.head()

wrote C:\Users\clemettn\Documents\phd\data_processed\08_casestudy_structure_datasets\ec8_gen2_site_specific_cbfs.pickle  (60 rows)


,name,n_storeys,roof_height,V_wind_GQWSI_LC1,V_wind_GQWSI_LC2,V_wind_GQWSI_LC3,T1_GQWSI_LC1,T1_GQWSI_LC2,T1_GQWSI_LC3,gravity_frame_elastic_baseshear,...,seismic_V4,seismic_F5,seismic_V5,seismic_F6,seismic_V6,seismic_F7,seismic_V7,site_idx,lat,lon
0,3s_cbf_dc2_site29,3,10500,19855.845357,33093.075595,19855.845357,0.898649,0.878176,0.889505,114036.402071,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29,40.9,-8.51787
1,3s_cbf_dc2_site4,3,10500,19855.845357,33093.075595,19855.845357,0.898649,0.878176,0.889505,116448.607363,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,37.5,40.88213
2,3s_cbf_dc2_site8,3,10500,19855.845357,33093.075595,19855.845357,0.898649,0.878176,0.889505,121599.903426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8,47.1,15.48213
3,3s_cbf_dc2_site19,3,10500,19855.845357,33093.075595,19855.845357,0.898649,0.878176,0.889505,153892.995259,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,40.3,14.98213
4,3s_cbf_dc2_site27,3,10500,19855.845357,33093.075595,19855.845357,0.898649,0.878176,0.889505,163844.024471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,44.0,22.88213


## 4. Build the MDOF analysis folders (structural model + modal + cyclic pushover + IDA)

One MDOF folder per site at `DEST_ROOT/site_{ii}/mdof/`, populated with the shared model,
a modal config, a FEMA 461 cyclic pushover config, and a FEMA P695 IDA (the 22-record
far-field set).

The MSA files are **not** added here — `050` adds them once the record selection exists.

In [28]:
# Name of the structural model file written into each analysis folder. The configs
# import the model by this name (their `model_file_name` variable), so it no longer
# has to be "structural_model.py".
REDUCED_RECORDERS = True

def add_cyclic_pushover_files(folder: Path, ctrl_node: int, key: str,
                              model_file_name: str) -> Path:
    """Add a FEMA 461 cyclic pushover run script + config to an analysis folder.
    The config imports the structural model from `model_file_name` in the folder itself.
    `key` (e.g. "site_12/mdof") selects any per-folder dU override and lets a tuned dU
    already on disk be preserved -- see phd_project/scripts/cpo_du.py."""
    copy_file(cfg["templates"]["run_cyclic_pushover"], folder / "run_cyclic_pushover.py")
    cfg_dst = folder / "config_cyclic_pushover.py"
    du = resolve_cpo_du(key, cfg_dst, CPO_DU_OVERRIDES, CPO_DU, PRESERVE_TUNED_DU)
    copy_analysis_config(
        cfg["templates"]["config_cyclic_pushover"],
        cfg_dst,
        results_folder_name="cyclic_pushover",
        model_file_name=model_file_name,
        update_config={
            "displacement_type": "displacement",
            "dU": du,
            "ctrl_node": ctrl_node,
            "displacements": CPO_DISPLACEMENTS,
        },
    )
    return cfg_dst


def add_ida_files(folder: Path, model_file_name: str) -> Path:
    """Add the FEMA P695 IDA run scripts + config to an analysis folder.
    Returns the IDA config path, which the ida.py batch launcher needs."""
    copy_file(cfg["templates"]["ida_process_recorder_roof_drift"],
              folder / "ida_process_recorders.py")
    copy_file(cfg["templates"]["config_im_SA"], folder / "config_im_SA.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"]["run_batch_ida_per_record"],
              folder / "run_batch_ida_per_record.py")
    copy_file(cfg["templates"]["run_ida_htf_per_record"],
              folder / "run_ida_htf_per_record.py")

    ida_config = folder / "config_ida_htf_femap695.py"
    copy_analysis_config(
        cfg["templates"]["config_ida_htf"],
        ida_config,
        results_folder_name="ida_femap695",
        model_file_name=model_file_name,
        gm_json_src_str=GM_JSON_SRC,
        record_filenames=FEMAP695_RECORDS,
        do_fill=False
    )
    return ida_config

def folder_format(site_idx: int):
    return f"site_{site_idx}" 
def build_mdof_folder(site_idx: int) -> Path:
    tag = site_tag[site_idx]
    folder = DEST_ROOT / folder_format(site_idx) / "mdof"
    folder.mkdir(parents=True, exist_ok=True)

    # design file (structural_model reads it from its own folder)
    design_out = Path(designs[tag]["design_out_json"])
    design_dst = folder / f"{tag}_designfile.json"
    copy_file(design_out, design_dst)

    # structural model # full recorders
    n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
    init_fn_full = copy_nlcbf_model(
        cfg["templates"],
        folder,
        design_json=design_dst.name,
        damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
    )
        
    # structural model # reduced recorders
    n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
    init_fn_reduced = copy_nlcbf_model(
        cfg["templates"],
        folder,
        design_json=design_dst.name,
        damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
        reduced=REDUCED_RECORDERS
    )

    # modal analysis config (fundamental period)
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        folder / "config_modal.py",
        results_folder_name="modal",
        model_file_name=init_fn_full,
        update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
    )

    # cyclic pushover (FEMA 461)
    add_cyclic_pushover_files(
        folder,
        get_control_node_from_design_file(design_dst),
        key=folder.relative_to(DEST_ROOT).as_posix(),
        model_file_name=init_fn_full,
    )

    # FEMA P695 IDA
    ida_config = add_ida_files(folder, model_file_name=init_fn_reduced)
    return folder, ida_config


mdof_folders = {folder_format(ii): build_mdof_folder(ii) for ii in tqdm(
    [ii for ii in all_sites if designs[site_tag[ii]]["success"]],
    desc="Building MDOF folders")}
print(f"built {len(mdof_folders)} MDOF folders")

Building MDOF folders:   0%|          | 0/60 [00:00<?, ?it/s]

built 60 MDOF folders


## 5. Batch launchers

Writes two launchers into
`cfg["scripts"]["batch_run_analyses"]/casestudy_sites/3s/mdof/` (the folder path encodes the
system `mdof` and storeys `3s`, so the filenames are just the analysis type):

| Launcher | Analysis | Template helper |
|---|---|---|
| `cpo.py` | Cyclic pushover (FEMA 461) | `configure_batch_run_file` |
| `ida.py` | FEMA P695 IDA (22 far-field records) | `copy_batch_ida_buildings` |

These write the launchers only — **run them separately** to get the results. The IDA is the
slow one (a multi-record IDA per site, run through the multi-building coordinator).

The MSA batch launchers live in `050_setup_msa_runs_for_complete_sites`, which writes one
launcher per (system, IM) for the sites that have a complete record set.

In [29]:
# --- batch launchers ----------------------------------------------------
BATCH_ROOT.mkdir(parents=True, exist_ok=True)

def _jobs(script_name, config_name, suffix):
    return [{"script": folder / script_name,
             "config": [folder / config_name],
             "name": [f"{tag}_{suffix}"]}
            for tag, (folder, _ida_config) in mdof_folders.items()]

# po / cpo / modal use the heterogeneous-jobs launcher template
for filename, jobs in [
    ("cpo.py", _jobs("run_cyclic_pushover.py", "config_cyclic_pushover.py", "cpo")),
    ("modal.py", _jobs("run_modal.py", "config_modal.py", "modal")),
]:
    configure_batch_run_file(cfg["templates"]["batch_run"], BATCH_ROOT / filename, jobs)
    print(f"wrote {filename} ({len(jobs)} jobs)")

# ida uses the multi-building coordinator launcher (a different template)
ida_buildings = [{"folder": folder, "config": ida_config}
                    for folder, ida_config in mdof_folders.values()]
copy_batch_ida_buildings(
    cfg["templates"]["run_batch_ida_buildings"],
    BATCH_ROOT / "ida.py",
    ida_buildings,
)
print(f"wrote ida.py ({len(ida_buildings)} buildings)")

print(f"\nall launchers -> {BATCH_ROOT}")
    

wrote cpo.py (60 jobs)
wrote modal.py (60 jobs)
wrote ida.py (60 buildings)

all launchers -> C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\batch_run_analyses\casestudy_sites\3s\mdof
